# 3-6. Argo CD GitOps와 KServe 배포 확인 결과 실습

이 노트북의 확인 질문은 `risk-classifier@candidate`가 GitOps 배포 경로에서 어디까지 검증됐는가입니다. API framework 호출을 반복하는 실습이 아니라, MLflow candidate가 Argo CD `Application`과 KServe `InferenceService`를 통해 endpoint 후보가 되는 증거를 확인합니다.


## 1. 실행 환경과 확인 결과 범위

이 셀에서는 live cluster 없이도 inspection 확인 결과를 만들 수 있는지 확인하는 것입니다. `kubectl`, `argocd`, KServe가 준비된 환경이면 live sync로 확장할 수 있지만, 이 노트북은 먼저 repository 안의 manifest를 구조화해서 읽습니다.


In [1]:
from __future__ import annotations

from pathlib import Path
import subprocess
from typing import Any

import utils as runtime

try:
    import yaml
except ModuleNotFoundError:
    import piplite

    await piplite.install("pyyaml")
    import yaml

prepared = await runtime.prepare_notebook()
ROOT = Path.cwd()
MANIFESTS = {
    "argocd_application": [
        "demos/ch03_docker_kubernetes/argocd/application.yaml",
        "artifacts/deployment/chapter_03/argocd/application.yaml",
    ],
    "gitops_overlay": [
        "demos/ch03_docker_kubernetes/argocd-resources/overlays/student/kustomization.yaml",
        "artifacts/deployment/chapter_03/argocd-resources/overlays/student/kustomization.yaml",
    ],
    "kserve_base": [
        "demos/ch03_docker_kubernetes/argocd-resources/base/inferenceservice.yaml",
        "artifacts/deployment/chapter_03/argocd-resources/base/inferenceservice.yaml",
    ],
    "observability_config": [
        "demos/ch03_docker_kubernetes/argocd-resources/base/observability-config.yaml",
        "artifacts/deployment/chapter_03/argocd-resources/base/observability-config.yaml",
    ],
    "ingress_host_patch": [
        "demos/ch03_docker_kubernetes/argocd-resources/overlays/student/ingress-host-patch.yaml",
        "artifacts/deployment/chapter_03/argocd-resources/overlays/student/ingress-host-patch.yaml",
    ],
}


def resolve_any(relative_paths: list[str]) -> tuple[Path, str]:
    errors: list[str] = []
    for relative_path in relative_paths:
        try:
            return runtime.resolve_course_path(relative_path), relative_path
        except FileNotFoundError as exc:
            errors.append(str(exc))
    raise FileNotFoundError(" | ".join(errors))


def read_yaml(paths: list[str]) -> dict[str, Any]:
    path, _ = resolve_any(paths)
    return yaml.safe_load(path.read_text(encoding="utf-8"))


def show_items(title: str, rows: list[tuple[str, object]]) -> None:
    print(f"\n[{title}]")
    for key, value in rows:
        print(f"- {key}: {value}")


def show_checks(title: str, rows: list[tuple[str, str, str]]) -> None:
    print(f"\n[{title}]")
    for check, status, meaning in rows:
        print(f"- {check}: {status} | {meaning}")

print("[manifest files]")
for name, candidates in MANIFESTS.items():
    try:
        path, selected = resolve_any(candidates)
        print(f"- {name}: found | {selected}")
    except FileNotFoundError:
        print(f"- {name}: missing | {' | '.join(candidates)}")


[manifest files]
- argocd_application: found | demos/ch03_docker_kubernetes/argocd/application.yaml
- gitops_overlay: found | demos/ch03_docker_kubernetes/argocd-resources/overlays/student/kustomization.yaml
- kserve_base: found | demos/ch03_docker_kubernetes/argocd-resources/base/inferenceservice.yaml
- observability_config: found | demos/ch03_docker_kubernetes/argocd-resources/base/observability-config.yaml
- ingress_host_patch: found | demos/ch03_docker_kubernetes/argocd-resources/overlays/student/ingress-host-patch.yaml


이 출력에서 확인할 핵심은 실습 확인 결과의 출처입니다. 다섯 파일이 모두 `found`여야 GitOps 배포 파일 inspection을 진행할 수 있습니다. 파일이 없으면 배포 실습 문제가 아니라 course artifact 누락 문제로 기록합니다.


## 2. Argo CD Application inspection

이 셀에서는 Argo CD가 어떤 Git 경로와 namespace를 배포 기준으로 삼는지 확인하는 것입니다. `repoURL`이 placeholder이면 live sync는 하지 않고, Application 구조 inspection까지만 보고합니다.


In [2]:
application = read_yaml(MANIFESTS["argocd_application"])
source = application["spec"]["source"]
destination = application["spec"]["destination"]
sync_options = application["spec"].get("syncPolicy", {}).get("syncOptions", [])

show_items(
    "Argo CD Application",
    [
        ("name", application["metadata"]["name"]),
        ("repoURL", source["repoURL"]),
        ("targetRevision", source["targetRevision"]),
        ("source.path", source["path"]),
        ("destination.namespace", destination["namespace"]),
        ("syncOptions", ", ".join(sync_options)),
    ],
)

if source["repoURL"] == "REPLACE_WITH_COURSE_GIT_REPO_URL":
    print("repoURL_status=placeholder; live_sync_evidence=unverified")
else:
    print("repoURL_status=configured; argocd_diff_sync_check=available")



[Argo CD Application]
- name: ai-quality-risk-classifier
- repoURL: REPLACE_WITH_COURSE_GIT_REPO_URL
- targetRevision: HEAD
- source.path: demos/ch03_docker_kubernetes/argocd-resources/overlays/student
- destination.namespace: ai-quality
- syncOptions: CreateNamespace=true, ApplyOutOfSyncOnly=true
repoURL_status=placeholder; live_sync_evidence=unverified


이 표에서 확인할 핵심은 `source.path`입니다. Argo CD가 `demos/ch03_docker_kubernetes/argocd-resources/overlays/student`가 아닌 다른 경로를 바라보면 후보 모델 endpoint가 아닌 다른 manifest를 동기화할 수 있습니다.


## 3. Git repository 연결과 ingress host 준비

이 셀에서는 live sync 전에 사람이 준비해야 하는 값을 확인합니다. Argo CD는 Git repository를 읽어 cluster에 적용하므로, 수강생 repository URL과 읽기 권한이 필요합니다. 또한 MLflow UI ingress host는 실습 환경마다 다르기 때문에 placeholder가 남아 있으면 live sync 준비가 끝난 것이 아닙니다.


In [3]:
ingress_patch_path, ingress_patch_selected = resolve_any(MANIFESTS["ingress_host_patch"])
ingress_patch = yaml.safe_load(ingress_patch_path.read_text(encoding="utf-8"))
ingress_host = next(item["value"] for item in ingress_patch if item.get("path") == "/spec/rules/0/host")
ingress_placeholder = "REPLACE_WITH_YOUR_INGRESS_DOMAIN" in ingress_host

show_items(
    "Live sync 전 수강생 수정값",
    [
        ("git repository URL", source["repoURL"]),
        ("Argo CD source.path", source["path"]),
        ("ingress patch file", ingress_patch_selected),
        ("MLflow ingress host", ingress_host),
    ],
)

print("ingress_host_status=placeholder" if ingress_placeholder else "ingress_host_status=configured")
print()
print("[실행 순서]")
print("1. bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh check")
print("2. edit demos/ch03_docker_kubernetes/argocd-resources/overlays/student/ingress-host-patch.yaml")
print("3. bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh key")
print("4. add the printed public key to GitHub Deploy keys")
print("5. ARGOCD_REPO_URL=git@github.com:<your-org-or-user>/<your-repo>.git bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh connect")
print("6. bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh sync")



[Live sync 전 수강생 수정값]
- git repository URL: REPLACE_WITH_COURSE_GIT_REPO_URL
- Argo CD source.path: demos/ch03_docker_kubernetes/argocd-resources/overlays/student
- ingress patch file: demos/ch03_docker_kubernetes/argocd-resources/overlays/student/ingress-host-patch.yaml
- MLflow ingress host: mlflow.REPLACE_WITH_YOUR_INGRESS_DOMAIN.example.com
ingress_host_status=placeholder

[실행 순서]
1. bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh check
2. edit demos/ch03_docker_kubernetes/argocd-resources/overlays/student/ingress-host-patch.yaml
3. bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh key
4. add the printed public key to GitHub Deploy keys
5. ARGOCD_REPO_URL=git@github.com:<your-org-or-user>/<your-repo>.git bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh connect
6. bash demos/ch03_docker_kubernetes/scripts/00_setup_argocd_gitops.sh sync


출력에서 `repoURL`이 placeholder이면 아직 Argo CD가 어떤 Git repository를 읽을지 모릅니다. `MLflow ingress host`에 placeholder가 남아 있으면 MLflow UI 주소도 아직 정해지지 않았습니다. 두 값 중 하나라도 준비되지 않았다면 sync 성공을 주장하지 않고, `manifest inspection`까지만 확인했다고 씁니다.


## 4. KServe 배포 파일 inspection

이 셀에서는 KServe `InferenceService`가 MLflow candidate와 연결되어 있는지 확인하는 것입니다. `InferenceService`는 endpoint 의도를 선언하지만, 실제 추론은 model format에 맞는 runtime container가 수행합니다.


In [4]:
inference_service = read_yaml(MANIFESTS["kserve_base"])
metadata = inference_service["metadata"]
sklearn = inference_service["spec"]["predictor"]["sklearn"]
annotations = metadata.get("annotations", {})
labels = metadata.get("labels", {})

show_items(
    "KServe InferenceService",
    [
        ("kind", inference_service["kind"]),
        ("model_alias", labels.get("ai-quality/model-alias")),
        ("mlflow_model_uri", annotations.get("ai-quality/mlflow-model-uri")),
        ("protocolVersion", sklearn["protocolVersion"]),
        ("storageUri", sklearn["storageUri"]),
        ("response_log_fields", annotations.get("ai-quality/observability-contract")),
    ],
)

print("kserve_control_plane=InferenceService; data_plane=Kubernetes Pod and ServingRuntime")



[KServe InferenceService]
- kind: InferenceService
- model_alias: candidate
- mlflow_model_uri: models:/risk-classifier@candidate
- protocolVersion: v2
- storageUri: pvc://ai-quality-models/artifacts/risk-classifier/candidate
- response_log_fields: request_id,model_version,score,threshold,prediction,latency_ms,validation_failure
kserve_control_plane=InferenceService; data_plane=Kubernetes Pod and ServingRuntime


이 출력에서 확인할 핵심은 KServe manifest가 추상화만 있는 파일이 아니라는 점입니다. `storageUri`와 model alias가 맞아야 runtime container가 의도한 후보 모델 artifact를 로딩할 수 있습니다.


## 5. Observability contract inspection

이 셀에서는 4장에서 찾을 telemetry field가 배포 manifest에 명시되어 있는지 확인하는 것입니다. Argo CD sync와 KServe Ready가 있어도 이 필드가 없으면 배포 확인에서 endpoint 확인 결과를 설명하기 어렵습니다.


In [5]:
observability = read_yaml(MANIFESTS["observability_config"])
telemetry_fields = [field.strip() for field in observability["data"]["TELEMETRY_FIELDS"].split(",")]

show_items(
    "Serving config",
    [
        ("MODEL_ALIAS", observability["data"].get("MODEL_ALIAS")),
        ("MODEL_VERSION", observability["data"].get("MODEL_VERSION")),
        ("DEPLOYMENT_CHECK", observability["data"].get("DEPLOYMENT_CHECK")),
    ],
)

print("\n[4장에서 다시 확인할 응답/로그 필드]")
for field in telemetry_fields:
    print(f"- {field}")

print("telemetry_contract_fields=" + ",".join(telemetry_fields))



[Serving config]
- MODEL_ALIAS: candidate
- MODEL_VERSION: candidate-v2
- DEPLOYMENT_CHECK: candidate_model_deployment

[4장에서 다시 확인할 응답/로그 필드]
- request_id
- model_version
- model_alias
- score
- threshold
- prediction
- latency_ms
- validation_failure
telemetry_contract_fields=request_id,model_version,model_alias,score,threshold,prediction,latency_ms,validation_failure


이 표에서 확인할 핵심은 배포 확인이 요구하는 필드가 명시되어 있다는 점입니다. `request_id`, `model_version`, `score`, `threshold`, `prediction`, `latency_ms`, `validation_failure`는 4장에서 로그와 metric으로 다시 확인합니다.


## 6. Live sync 가능성 확인

이 셀에서는 지금 환경에서 live Argo CD sync를 주장할 수 있는지 확인하는 것입니다. CLI가 없거나 repository URL이 placeholder이면 live sync는 미확인으로 두고, manifest inspection 확인 결과만 보고합니다.


In [6]:
def command_available(command: str) -> bool:
    return subprocess.run(["which", command], capture_output=True, text=True).returncode == 0

repo_url = source["repoURL"]
live_readiness = [
    ("repoURL_configured", "pass" if not repo_url.startswith("REPLACE_WITH") else "blocked", "실제 Git repository URL 필요"),
    ("ingress_host_configured", "pass" if not ingress_placeholder else "blocked", "수강생별 MLflow ingress 주소 필요"),
    ("kubectl_cli", "pass" if command_available("kubectl") else "blocked", "Application 등록과 KServe status 확인 필요"),
    ("argocd_cli", "pass" if command_available("argocd") else "blocked", "diff/sync/wait 실행 필요"),
    ("kustomize_cli", "pass" if command_available("kustomize") else "optional", "overlay render 확인 가능"),
]

show_checks("live 확인 준비 상태", live_readiness)

if any(status == "blocked" for _, status, _ in live_readiness):
    print("live_sync_evidence=unverified; evidence_scope=manifest_inspection")
else:
    print("live_sync_evidence=checkable; required_checks=argocd_diff,argocd_sync,argocd_wait,kserve_ready")



[live 확인 준비 상태]
- repoURL_configured: blocked | 실제 Git repository URL 필요
- ingress_host_configured: blocked | 수강생별 MLflow ingress 주소 필요
- kubectl_cli: pass | Application 등록과 KServe status 확인 필요
- argocd_cli: pass | diff/sync/wait 실행 필요
- kustomize_cli: pass | overlay render 확인 가능
live_sync_evidence=unverified; evidence_scope=manifest_inspection


이 출력에서 확인할 핵심은 blocker를 숨기지 않는 것입니다. live sync가 막힌 경우 “배포 성공”이라고 쓰지 않고, `manifest inspection completed, live sync unverified`라고 기록해야 합니다.


## 7. 배포 확인 결과 작성

이 셀에서는 3장에서 4장으로 넘길 확인 결과 행를 만드는 것입니다. 이 row는 4장 observability notebook과 5장 배포 확인 notebook에서 그대로 사용합니다.


In [7]:
live_sync_status = "verified" if all(status in {"pass", "optional"} for _, status, _ in live_readiness) else "unverified"

handoff = [
    ("check_scope", "후보 모델 배포 확인", "보고서 범위"),
    ("model_uri", annotations.get("ai-quality/mlflow-model-uri"), "inspection"),
    ("gitops_path", source["path"], "inspection"),
    ("mlflow_ingress_host", ingress_host, "student-edit"),
    ("argocd_app", application["metadata"]["name"], "inspection"),
    ("kserve_resource", "InferenceService/ai-quality-risk-classifier-dev", "inspection"),
    ("telemetry_fields", ", ".join(telemetry_fields), "4장 확인 대상"),
    ("live_sync", live_sync_status, "environment-dependent"),
]

print("[4장으로 넘길 확인 항목]")
for field, value, status in handoff:
    print(f"- {field}: {value} ({status})")


[4장으로 넘길 확인 항목]
- check_scope: 후보 모델 배포 확인 (보고서 범위)
- model_uri: models:/risk-classifier@candidate (inspection)
- gitops_path: demos/ch03_docker_kubernetes/argocd-resources/overlays/student (inspection)
- mlflow_ingress_host: mlflow.REPLACE_WITH_YOUR_INGRESS_DOMAIN.example.com (student-edit)
- argocd_app: ai-quality-risk-classifier (inspection)
- kserve_resource: InferenceService/ai-quality-risk-classifier-dev (inspection)
- telemetry_fields: request_id, model_version, model_alias, score, threshold, prediction, latency_ms, validation_failure (4장 확인 대상)
- live_sync: unverified (environment-dependent)


이 확인 결과 행으로 쓸 수 있는 보고서 문장은 다음과 같습니다.

`risk-classifier@candidate`는 Argo CD Application `ai-quality-risk-classifier`의 GitOps path `demos/ch03_docker_kubernetes/argocd-resources/overlays/student`에 선언되어 있고, KServe `InferenceService`는 candidate storage URI와 응답/로그 필드를 포함합니다. MLflow ingress host는 수강생 환경에 맞게 수정되어야 하며, 현재 환경의 live sync 상태는 `verified/unverified`로 구분해 4장 observability 확인 결과와 5장 배포 확인에서 다시 확인합니다.
